PROJECT 2:
Pakistani Politician Image Classification using CNN and ResNet50

Dataset:
• 16 Classes (Politicians)
• Train / Validation / Test Split

Team Members:

• Mashal → Dataset Collection + ResNet50 (Transfer Learning)+ Evaluation

• Ayesha → CNN Model + Preprocessing + Base Training

• Shiza → Data Augmentation + Graphs + Results Visualization

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 🔹 Importing Required Libraries
We import TensorFlow, NumPy, Matplotlib, Seaborn, and Scikit-learn tools for building and evaluating CNN and ResNet50 models.

In [3]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

from sklearn.metrics import confusion_matrix, classification_report

## 🔹 Dataset Paths Setup
We define the directory paths for training, validation, and testing datasets stored in Google Drive.

In [4]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

base_dir = "/content/drive/MyDrive/Dataset/dataset_split_final"

train_dir = f"{base_dir}/train_final"
val_dir   = f"{base_dir}/val_final"
test_dir  = f"{base_dir}/test_final"

print("Train Path:", train_dir)
print("Validation Path:", val_dir)
print("Test Path:", test_dir)

Train Path: /content/drive/MyDrive/Dataset/dataset_split_final/train_final
Validation Path: /content/drive/MyDrive/Dataset/dataset_split_final/val_final
Test Path: /content/drive/MyDrive/Dataset/dataset_split_final/test_final


##  Data Augmentation & Generators
To improve model generalization, we apply image augmentation such as rotation, zoom, brightness variation, and flipping.

In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Augmentation ONLY for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1
)

val_test_datagen = ImageDataGenerator(rescale=1./255)  # NO augmentation

train_gen = train_datagen.flow_from_directory(
    train_dir, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True
)
val_gen = val_test_datagen.flow_from_directory(
    val_dir, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    test_dir, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

NUM_CLASSES = train_gen.num_classes
CLASS_NAMES = list(train_gen.class_indices.keys())
print(f"Classes ({NUM_CLASSES}):", CLASS_NAMES)

Found 1297 images belonging to 16 classes.
Found 257 images belonging to 16 classes.
Found 175 images belonging to 16 classes.
Classes (16): ['Benazir_Bhutto', 'Bilawal Bhutto', 'Imran_khan', 'Ishaq_Dar', 'Khawaja_Asif', 'Maryam Nawaz', 'Mohsin naqvi', 'Murad_Saeed', 'Nawaz_Sharif', 'Qamar_Javed_Bajwa', 'Shafqat_Mehmood', 'Shah_Mahmood_Qureshi', 'Shehbaz_sharif', 'Sheikh Rasheed Ahmed', 'Shireen_Mazari', 'asif_zardari']


 Model Builder Function

In [6]:
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

def build_model(base_model_fn, input_shape=(224, 224, 3), num_classes=16, name="model"):
    base = base_model_fn(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = False  # Freeze first

    inputs = tf.keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs, name=name)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model, base

resnet_model, resnet_base = build_model(ResNet50, num_classes=NUM_CLASSES, name="ResNet50")
efficientnet_model, eff_base = build_model(EfficientNetB0, num_classes=NUM_CLASSES, name="EfficientNetB0")

print("ResNet50 summary:")
resnet_model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
ResNet50 summary:


Model: "ResNet50"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │         2,064 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,155,408 (92.15 MB)

 Trainable params: 563,600 (2.15 MB)

 Non-trainable params: 23,591,808 (90.00 MB)

Training Function (Phase 1: Frozen Base)

In [7]:
def get_callbacks(model_name):
    return [
        EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, verbose=1),
        ModelCheckpoint(f'/content/drive/MyDrive/{model_name}_best.h5',
                        save_best_only=True, monitor='val_accuracy', verbose=1)
    ]

EPOCHS_PHASE1 = 20

print("=" * 50)
print("Training ResNet50 - Phase 1 (Frozen base)")
print("=" * 50)
resnet_history1 = resnet_model.fit(
    train_gen, epochs=EPOCHS_PHASE1,
    validation_data=val_gen,
    callbacks=get_callbacks("resnet50")
)

Training ResNet50 - Phase 1 (Frozen base)
Epoch 1/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 14s/step - accuracy: 0.1209 - loss: 2.8200 
Epoch 1: val_accuracy improved from None to 0.07393, saving model to /content/drive/MyDrive/resnet50_best.h5



Epoch 1: finished saving model to /content/drive/MyDrive/resnet50_best.h5
41/41 ━━━━━━━━━━━━━━━━━━━━ 738s 18s/step - accuracy: 0.1388 - loss: 2.7749 - val_accuracy: 0.0739 - val_loss: 2.6890 - learning_rate: 0.0010
Epoch 2/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.1503 - loss: 2.6473
Epoch 2: val_accuracy improved from 0.07393 to 0.11673, saving model to /content/drive/MyDrive/resnet50_best.h5



Epoch 2: finished saving model to /content/drive/MyDrive/resnet50_best.h5
41/41 ━━━━━━━━━━━━━━━━━━━━ 328s 8s/step - accuracy: 0.1704 - loss: 2.6130 - val_accuracy: 0.1167 - val_loss: 2.6326 - learning_rate: 0.0010
Epoch 3/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.1687 - loss: 2.5883
Epoch 3: val_accuracy did not improve from 0.11673
41/41 ━━━━━━━━━━━━━━━━━━━━ 324s 8s/step - accuracy: 0.1804 - loss: 2.5915 - val_accuracy: 0.1012 - val_loss: 2.6221 - learning_rate: 0.0010
Epoch 4/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.2126 - loss: 2.4709
Epoch 4: val_accuracy improved from 0.11673 to 0.12451, saving model to /content/drive/MyDrive/resnet50_best.h5



Epoch 4: finished saving model to /content/drive/MyDrive/resnet50_best.h5
41/41 ━━━━━━━━━━━━━━━━━━━━ 378s 8s/step - accuracy: 0.2113 - loss: 2.5031 - val_accuracy: 0.1245 - val_loss: 2.6176 - learning_rate: 0.0010
Epoch 5/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.2246 - loss: 2.4459
Epoch 5: val_accuracy improved from 0.12451 to 0.18288, saving model to /content/drive/MyDrive/resnet50_best.h5



Epoch 5: finished saving model to /content/drive/MyDrive/resnet50_best.h5
41/41 ━━━━━━━━━━━━━━━━━━━━ 327s 8s/step - accuracy: 0.2136 - loss: 2.4811 - val_accuracy: 0.1829 - val_loss: 2.5927 - learning_rate: 0.0010
Epoch 6/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.2465 - loss: 2.3969
Epoch 6: val_accuracy improved from 0.18288 to 0.24514, saving model to /content/drive/MyDrive/resnet50_best.h5



Epoch 6: finished saving model to /content/drive/MyDrive/resnet50_best.h5
41/41 ━━━━━━━━━━━━━━━━━━━━ 353s 9s/step - accuracy: 0.2390 - loss: 2.4311 - val_accuracy: 0.2451 - val_loss: 2.5685 - learning_rate: 0.0010
Epoch 7/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.2456 - loss: 2.3935
Epoch 7: val_accuracy did not improve from 0.24514
41/41 ━━━━━━━━━━━━━━━━━━━━ 328s 8s/step - accuracy: 0.2344 - loss: 2.4152 - val_accuracy: 0.2451 - val_loss: 2.5192 - learning_rate: 0.0010
Epoch 8/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.2613 - loss: 2.3482
Epoch 8: val_accuracy improved from 0.24514 to 0.27626, saving model to /content/drive/MyDrive/resnet50_best.h5



Epoch 8: finished saving model to /content/drive/MyDrive/resnet50_best.h5
41/41 ━━━━━━━━━━━━━━━━━━━━ 321s 8s/step - accuracy: 0.2452 - loss: 2.3928 - val_accuracy: 0.2763 - val_loss: 2.4969 - learning_rate: 0.0010
Epoch 9/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.2552 - loss: 2.3717
Epoch 9: val_accuracy improved from 0.27626 to 0.30739, saving model to /content/drive/MyDrive/resnet50_best.h5



Epoch 9: finished saving model to /content/drive/MyDrive/resnet50_best.h5
41/41 ━━━━━━━━━━━━━━━━━━━━ 327s 8s/step - accuracy: 0.2567 - loss: 2.3771 - val_accuracy: 0.3074 - val_loss: 2.4339 - learning_rate: 0.0010
Epoch 10/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.2741 - loss: 2.3326
Epoch 10: val_accuracy did not improve from 0.30739
41/41 ━━━━━━━━━━━━━━━━━━━━ 353s 9s/step - accuracy: 0.2683 - loss: 2.3634 - val_accuracy: 0.2879 - val_loss: 2.3955 - learning_rate: 0.0010
Epoch 11/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.2413 - loss: 2.3266
Epoch 11: val_accuracy did not improve from 0.30739
41/41 ━━━━━━━━━━━━━━━━━━━━ 322s 8s/step - accuracy: 0.2406 - loss: 2.3262 - val_accuracy: 0.2918 - val_loss: 2.3429 - learning_rate: 0.0010
Epoch 12/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.2556 - loss: 2.3000
Epoch 12: val_accuracy did not improve from 0.30739
41/41 ━━━━━━━━━━━━━━━━━━━━ 317s 8s/step - accuracy: 0.2583 - loss: 2.3216 - val_accuracy: 0.3074 


Epoch 13: finished saving model to /content/drive/MyDrive/resnet50_best.h5
41/41 ━━━━━━━━━━━━━━━━━━━━ 326s 8s/step - accuracy: 0.2652 - loss: 2.2865 - val_accuracy: 0.3424 - val_loss: 2.2504 - learning_rate: 0.0010
Epoch 14/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.2492 - loss: 2.3188
Epoch 14: val_accuracy did not improve from 0.34241
41/41 ━━━━━━━━━━━━━━━━━━━━ 377s 8s/step - accuracy: 0.2444 - loss: 2.3272 - val_accuracy: 0.3346 - val_loss: 2.2463 - learning_rate: 0.0010
Epoch 15/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.2961 - loss: 2.2221
Epoch 15: val_accuracy did not improve from 0.34241
41/41 ━━━━━━━━━━━━━━━━━━━━ 314s 8s/step - accuracy: 0.2907 - loss: 2.2272 - val_accuracy: 0.3152 - val_loss: 2.1973 - learning_rate: 0.0010
Epoch 16/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.2955 - loss: 2.2477
Epoch 16: val_accuracy did not improve from 0.34241
41/41 ━━━━━━━━━━━━━━━━━━━━ 318s 8s/step - accuracy: 0.2868 - loss: 2.2530 - val_accuracy: 0.3424


Epoch 17: finished saving model to /content/drive/MyDrive/resnet50_best.h5
41/41 ━━━━━━━━━━━━━━━━━━━━ 317s 8s/step - accuracy: 0.2791 - loss: 2.2256 - val_accuracy: 0.3658 - val_loss: 2.1273 - learning_rate: 0.0010
Epoch 18/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.3157 - loss: 2.1777
Epoch 18: val_accuracy did not improve from 0.36576
41/41 ━━━━━━━━━━━━━━━━━━━━ 348s 8s/step - accuracy: 0.3076 - loss: 2.2035 - val_accuracy: 0.3619 - val_loss: 2.0923 - learning_rate: 0.0010
Epoch 19/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.2998 - loss: 2.1621
Epoch 19: val_accuracy did not improve from 0.36576
41/41 ━━━━━━━━━━━━━━━━━━━━ 348s 8s/step - accuracy: 0.3022 - loss: 2.1799 - val_accuracy: 0.3541 - val_loss: 2.0616 - learning_rate: 0.0010
Epoch 20/20
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.3117 - loss: 2.1805
Epoch 20: val_accuracy did not improve from 0.36576
41/41 ━━━━━━━━━━━━━━━━━━━━ 313s 8s/step - accuracy: 0.2984 - loss: 2.1931 - val_accuracy: 0.3541

In [8]:
print(resnet_history1.history['val_accuracy'][-1])  # should print 0.365...

0.3540855944156647
